In [1]:
%cd /content
!git clone https://github.com/fmaignacio/datajud_probe.git datajud_probe
%cd /content/datajud_probe


/content
Cloning into 'datajud_probe'...
remote: Enumerating objects: 190, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 190 (delta 100), reused 169 (delta 79), pack-reused 0 (from 0)
Receiving objects: 100% (190/190), 2.79 MiB | 6.71 MiB/s, done.
Resolving deltas: 100% (100/100), done.
/content/datajud_probe


# 04 - Parse tabela de assuntos

Objetivo: transformar a tabela de assuntos processuais exportada em `.xls`/HTML em um dicionario reutilizavel de codigos, rotulos e caminhos hierarquicos.

A saida principal e `data/reference/assuntos/processed/assuntos_lookup.parquet`, usada pelo notebook `01_exploracao_stj_metadados.ipynb` para trocar codigos como `03608` por rotulos como `Trafico de Drogas e Condutas Afins`.

## 1. Ambiente

Este notebook deve ser executado preferencialmente no ambiente local do projeto, porque a tabela `.xls` esta versionada no repositorio. Depois, se a EDA for rodada no Colab, copie os arquivos gerados em `data/reference/assuntos/processed/` para o Drive em `MyDrive/Mestrado/2026/llms/data/reference/assuntos/processed/`.

In [2]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("/Users/felipeignacio/Projects/datajud_probe")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.assuntos import add_hierarchy_paths, parse_assuntos_files, save_assuntos_lookup

PROJECT_ROOT

PosixPath('/content/datajud_probe')

## 2. Localizar tabelas de assuntos

Por enquanto ha uma tabela de `Justica Federal 1º Grau`. Se depois forem adicionadas tabelas de outras instancias, basta colocar os arquivos em `data/reference/assuntos/raw/` ou em `notebooks/` com nome contendo `Tabela_Assuntos`.

In [3]:
INPUT_DIRS = [
    PROJECT_ROOT / "data" / "reference" / "assuntos" / "raw",
    PROJECT_ROOT / "notebooks",
]

subject_files = []
for input_dir in INPUT_DIRS:
    if input_dir.exists():
        subject_files.extend(input_dir.glob("*Tabela_Assuntos*.xls"))

subject_files = sorted(set(subject_files))

print(f"Tabelas encontradas: {len(subject_files)}")
for path in subject_files:
    print("-", path.relative_to(PROJECT_ROOT))

if not subject_files:
    raise FileNotFoundError("Nenhuma tabela de assuntos encontrada.")

Tabelas encontradas: 1
- notebooks/78_Tabela_Assuntos_Justica_Federal_1_Grau.xls


## 3. Parsear e normalizar

A tabela parece `.xls`, mas o conteudo e HTML. O parser trata esse HTML diretamente, preservando codigo, codigo pai, glossario, ODS, datas e instancia de origem.

In [4]:
assuntos_df = parse_assuntos_files(subject_files)
assuntos_df = add_hierarchy_paths(assuntos_df)

print(assuntos_df.shape)
assuntos_df.head(10)

(2435, 17)


,assunto,codigo,codigo_pai,dispositivo_legal,artigo,alteracoes,glossario,ods,data_publicacao,data_alteracao,data_inativacao,data_reativacao,nivel_visual,fonte,instancia,caminho_codigos,caminho_assuntos
0,DIREITO À EDUCAÇÃO,12775,None,None,None,None,None,None,NaT,NaT,NaT,NaT,1,78_Tabela_Assuntos_Justica_Federal_1_Grau.xls,Justiça Federal 1º Grau,12775,DIREITO À EDUCAÇÃO
1,Acesso,12795,12775,CF; LDB,None,None,Dispõe sobre o acesso à educação;,"ODS 16 - Paz, justiça e instituições eficazes",2020-09-23,2021-07-14 11:18:51,NaT,NaT,1,78_Tabela_Assuntos_Justica_Federal_1_Grau.xls,Justiça Federal 1º Grau,12775 > 12795,DIREITO À EDUCAÇÃO > Acesso
2,Acesso sem Conclusão do Ensino Médio,12805,12795,LDB,"44, II",glossario;,Dispõe sobre o acesso ao ensino superior dos e...,"ODS 4 - Educação de qualidade; ODS 16 - Paz, j...",2020-09-23,2021-07-14 11:13:51,NaT,NaT,2,78_Tabela_Assuntos_Justica_Federal_1_Grau.xls,Justiça Federal 1º Grau,12775 > 12795 > 12805,DIREITO À EDUCAÇÃO > Acesso > Acesso sem Concl...
3,Ciência Sem Fronteiras,14177,12795,Lei 9394/1996; CF,"53, V e 208",None,Demandas em que se discute a exigência de apre...,"ODS 4 - Educação de qualidade; ODS 16 - Paz, j...",2021-01-18,NaT,NaT,NaT,2,78_Tabela_Assuntos_Justica_Federal_1_Grau.xls,Justiça Federal 1º Grau,12775 > 12795 > 14177,DIREITO À EDUCAÇÃO > Acesso > Ciência Sem Fron...
4,Cobrança de Taxa de Matrícula,12808,12795,Lei 9394/1996. CF,"Art. 53, V; 206, IV e 208",glossario;,Classificar demandas nas quais se discute a po...,"ODS 16 - Paz, justiça e instituições eficazes",2020-09-23,2021-01-18 13:09:35,NaT,NaT,2,78_Tabela_Assuntos_Justica_Federal_1_Grau.xls,Justiça Federal 1º Grau,12775 > 12795 > 12808,DIREITO À EDUCAÇÃO > Acesso > Cobrança de Taxa...
5,Convalidação de Estudos e Reconhecimento de Di...,12806,12795,LDB,"art. 48, caput, §1º, §3º, 53, VI e 80, §2º",glossario;,Dispõe sobre o reconhecimento de diplomas que ...,"ODS 4 - Educação de qualidade; ODS 16 - Paz, j...",2020-09-23,2021-07-14 11:13:20,NaT,NaT,2,78_Tabela_Assuntos_Justica_Federal_1_Grau.xls,Justiça Federal 1º Grau,12775 > 12795 > 12806,DIREITO À EDUCAÇÃO > Acesso > Convalidação de ...
6,Cota para Ingresso - Ações Afirmativas,12809,12795,Lei 12.711/2012,None,glossario;,Dispõe sobre a reserva de vagas para estudante...,"ODS 4 - Educação de qualidade; ODS 16 - Paz, j...",2020-09-23,2021-07-14 11:12:53,NaT,NaT,2,78_Tabela_Assuntos_Justica_Federal_1_Grau.xls,Justiça Federal 1º Grau,12775 > 12795 > 12809,DIREITO À EDUCAÇÃO > Acesso > Cota para Ingres...
7,Itinerários Formativos,12810,12795,LDB,None,glossario;,DDispõe sobre a composição de itinerários form...,"ODS 4 - Educação de qualidade; ODS 16 - Paz, j...",2020-09-23,2021-07-14 11:12:36,NaT,NaT,2,78_Tabela_Assuntos_Justica_Federal_1_Grau.xls,Justiça Federal 1º Grau,12775 > 12795 > 12810,DIREITO À EDUCAÇÃO > Acesso > Itinerários Form...
8,Formação Técnica e Profissional,12910,12810,CF; LDB,"36, V",glossario;,Dispõe sobre a formação técnica e profissional,"ODS 4 - Educação de qualidade; ODS 16 - Paz, j...",2020-09-23,2021-07-14 11:12:05,NaT,NaT,3,78_Tabela_Assuntos_Justica_Federal_1_Grau.xls,Justiça Federal 1º Grau,12775 > 12795 > 12810 > 12910,DIREITO À EDUCAÇÃO > Acesso > Itinerários Form...
9,Itinerários Formativos do Ensino Médio,12911,12810,LDB,Art. 36,glossario;,DDispõe sobre a composição de itinerários form...,"ODS 4 - Educação de qualidade; ODS 16 - Paz, j...",2020-10-01,2021-07-14 11:11:43,NaT,NaT,3,78_Tabela_Assuntos_Justica_Federal_1_Grau.xls,Justiça Federal 1º Grau,12775 > 12795 > 12810 > 12911,DIREITO À EDUCAÇÃO > Acesso > Itinerários Form...


In [5]:
# Checagem rapida do assunto mais recorrente na EDA inicial.
assuntos_df.loc[
    assuntos_df["codigo"].eq("03608"),
    ["codigo", "assunto", "codigo_pai", "caminho_assuntos", "instancia"],
]

,codigo,assunto,codigo_pai,caminho_assuntos,instancia
1632,03608,Tráfico de Drogas e Condutas Afins,03607,DIREITO PENAL > Crimes Previstos na Legislação...,Justiça Federal 1º Grau


## 4. Salvar lookup

O CSV facilita inspecao manual. O Parquet e preferido para leitura automatica no notebook de EDA.

In [6]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "reference" / "assuntos" / "processed"
outputs = save_assuntos_lookup(assuntos_df, OUTPUT_DIR)
outputs

{'csv': PosixPath('/content/datajud_probe/data/reference/assuntos/processed/assuntos_lookup.csv'),
 'parquet': PosixPath('/content/datajud_probe/data/reference/assuntos/processed/assuntos_lookup.parquet')}

In [7]:
# Amostra compacta para revisao visual.
assuntos_df[["codigo", "assunto", "codigo_pai", "caminho_assuntos"]].head(20)

,codigo,assunto,codigo_pai,caminho_assuntos
0,12775,DIREITO À EDUCAÇÃO,None,DIREITO À EDUCAÇÃO
1,12795,Acesso,12775,DIREITO À EDUCAÇÃO > Acesso
2,12805,Acesso sem Conclusão do Ensino Médio,12795,DIREITO À EDUCAÇÃO > Acesso > Acesso sem Concl...
3,14177,Ciência Sem Fronteiras,12795,DIREITO À EDUCAÇÃO > Acesso > Ciência Sem Fron...
4,12808,Cobrança de Taxa de Matrícula,12795,DIREITO À EDUCAÇÃO > Acesso > Cobrança de Taxa...
5,12806,Convalidação de Estudos e Reconhecimento de Di...,12795,DIREITO À EDUCAÇÃO > Acesso > Convalidação de ...
6,12809,Cota para Ingresso - Ações Afirmativas,12795,DIREITO À EDUCAÇÃO > Acesso > Cota para Ingres...
7,12810,Itinerários Formativos,12795,DIREITO À EDUCAÇÃO > Acesso > Itinerários Form...
8,12910,Formação Técnica e Profissional,12810,DIREITO À EDUCAÇÃO > Acesso > Itinerários Form...
9,12911,Itinerários Formativos do Ensino Médio,12810,DIREITO À EDUCAÇÃO > Acesso > Itinerários Form...


## 5. Uso na EDA

Depois de gerar o lookup, rode `01_exploracao_stj_metadados.ipynb`. Ele procura automaticamente por `assuntos_lookup.parquet` no repositorio local e, se estiver no Colab, tambem no Drive em `data/reference/assuntos/processed/`.